[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C66_Agentic_Evaluation_Course/01_benchmarks/01_benchmark_landscape.ipynb)

# 01 · agentic 基准全景（画像库 / 判分方式分类 / 任务区分度 / 污染检测 / 规模与分辨力）

目标：把「有哪些基准」这个背诵题，变成几个**可以查询、可以计算、可以复用**的工具。

本 notebook 你会亲手实现：
1. **基准画像库与查询** —— 十余个主流 agentic 基准的结构化画像，按环境/判分方式检索
2. **判分方式分类器** —— 给定一个任务描述，判断它该用哪种判分函数
3. **任务区分度与信息量** —— 为什么全对/全错的任务是 0 bit，成功率该往哪推
4. **n-gram 污染检测器** —— 纯 Python 实现，检出「任务文本与训练语料重合」
5. **时间截断检验** —— 用发布时间切分任务集，检出「训练截止前后的分数断崖」
6. **任务集规模与分辨力** —— 要区分差 3 个点的两个 agent，需要多少道题

> 心智模型：**基准不是排行榜，是测量仪器。选基准 = 选一把尺子，
> 而每把尺子都有自己的量程、刻度和系统误差。**

## 1 · 基准画像库与查询

先把讲解里的那张地图变成数据。字段的选择本身就是本模块的结论：
**环境类型决定判分方式，判分方式决定这个分数能支持什么结论。**

In [ ]:
import math, json, re
from collections import Counter, defaultdict
import numpy as np

# 说明：n_tasks 取各基准官方公布的量级，随版本滚动会变；报告分数时必须写明具体版本。
BENCHMARKS = [
    # name,                 env,        scoring,        n_tasks, reproducible, notes
    ('SWE-bench',           'repo',     'tests',        2294, 'high',  '自动化流水线，含不可解任务'),
    ('SWE-bench Verified',  'repo',     'tests',         500, 'high',  '人工逐条审核后的可解子集'),
    ('SWE-bench Lite',      'repo',     'tests',         300, 'high',  '轻量子集，适合 CI 回归'),
    ('SWE-bench Multimodal','repo',     'tests',         517, 'high',  'JS 仓库 + 截图/录屏'),
    ('Terminal-Bench',      'terminal', 'tests',         100, 'high',  '终端环境，测试脚本判分'),
    ('tau-bench',           'tools',    'final_state',   165, 'medium','用户由 LLM 模拟，注入方差'),
    ('tau2-bench',          'tools',    'final_state',   278, 'medium','双向控制：agent 指挥用户操作'),
    ('WebArena',            'web_self', 'validator',     812, 'high',  '自托管站点，含不可能任务'),
    ('VisualWebArena',      'web_self', 'validator',     910, 'high',  '需要视觉理解的网页任务'),
    ('OSWorld',             'os',       'validator',     369, 'medium','真实 OS + 真实应用，跨应用任务'),
    ('Mind2Web',            'web_off',  'element_match',2350, 'high',  '离线轨迹，无法评错误恢复'),
    ('GAIA',                'open_web', 'exact_match',   466, 'low',   '唯一短答案；互联网会漂移'),
    ('BrowseComp',          'open_web', 'exact_match',  1266, 'low',   '难找易验证，专测长程搜索'),
    ('MLE-bench',           'research', 'human_baseline', 75, 'medium','Kaggle 奖牌线做标尺'),
]

FIELDS = ['name', 'env', 'scoring', 'n_tasks', 'reproducible', 'notes']
DB = [dict(zip(FIELDS, row)) for row in BENCHMARKS]

def query(**kw):
    out = DB
    for k, v in kw.items():
        out = [b for b in out if b[k] == v]
    return out

print('按判分方式统计：')
for scoring, n in Counter(b['scoring'] for b in DB).most_common():
    names = ', '.join(b['name'] for b in query(scoring=scoring))
    print(f'  {scoring:<15} {n} 个 | {names}')

assert len(query(scoring='tests')) == 5
assert all(b['reproducible'] == 'low' for b in query(env='open_web'))
print('\n✅ 注意最后一条断言：所有开放网络基准的可复现性都是 low——')
print('   这不是巧合，是「用真实互联网当环境」的必然代价。')

In [ ]:
# 判分方式 → 该分数能支持什么结论
IMPLICATION = {
    'tests':          ('客观性最高', '结论仅限于「测试覆盖到的行为」，测试弱则判分弱'),
    'final_state':    ('客观性高',   '仅适用于有写操作的任务；纯咨询任务需额外的信息检查项'),
    'validator':      ('客观性高',   '每题一个手写函数 → validator 本身可能有 bug，需要被测试'),
    'element_match':  ('客观性高',   '离线：无法评「走错后能不能纠正」这一最关键的 agent 能力'),
    'exact_match':    ('客观性最高', '要求答案唯一简短 → 任务形态被判分方式严格约束'),
    'human_baseline': ('相对尺度',   '跨任务可聚合，但接近人类上限时区分度快速消失'),
}
w = max(len(k) for k in IMPLICATION)
for k, (pro, con) in IMPLICATION.items():
    print(f'{k:<{w}}  {pro:<8} ⚠ {con}')

total_tasks = sum(b['n_tasks'] for b in DB)
print(f'\n画像库覆盖 {len(DB)} 个基准、{total_tasks:,} 道任务。')
assert total_tasks > 10000
print('✅ 一个常被忽略的事实：agentic 基准的任务总量比静态基准少两三个数量级——')
print('   因为每道题都要一套环境和一个判分器，边际成本极高。这直接决定了统计功效的天花板（第 6 节）。')

## 2 · 判分方式分类器

自建任务集时的第一个决策：这道题该怎么判分？把讲解里的规则写成一棵可执行的决策树。

In [ ]:
def choose_scorer(has_tests, mutates_state, answer_is_unique_short, is_offline_trace,
                  has_human_baseline):
    """判分方式选择树。顺序即优先级：能跑测试就跑测试，其次看终态，
    再次看唯一答案，最后才考虑相对基线/人工。"""
    if has_tests:
        return 'tests'
    if mutates_state:
        return 'final_state'
    if answer_is_unique_short:
        return 'exact_match'
    if is_offline_trace:
        return 'element_match'
    if has_human_baseline:
        return 'human_baseline'
    return 'llm_judge'          # 兜底：交给 C67

CASES = [
    (True,  True,  False, False, False, '修一个有回归测试的 bug'),
    (False, True,  False, False, False, '帮用户改签机票'),
    (False, False, True,  False, False, '查出某届会议最佳论文一作的博士导师'),
    (False, False, False, True,  False, '在录制好的网页轨迹上复现一次下单'),
    (False, False, False, False, True,  '在一个 Kaggle 数据集上把 AUC 做到尽可能高'),
    (False, False, False, False, False, '写一份关于某市场的调研摘要'),
]
for args in CASES:
    print(f'{args[-1]:<32} -> {choose_scorer(*args[:-1])}')

assert choose_scorer(True, True, True, True, True) == 'tests'
assert choose_scorer(False, False, False, False, False) == 'llm_judge'
print('\n✅ 决策树就位。最后一条落到 llm_judge 上——这是唯一一类「判分器自己需要被评测」的情形，')
print('   本课把它整个交给 C67，因为它值一门独立的课。')

## 3 · 任务区分度与信息量：成功率该往哪推

一道所有 agent 都通过（或都失败）的题，对「谁更强」这个问题贡献 0 bit。
用二元熵 $H(p) = -p\log_2 p - (1-p)\log_2(1-p)$ 量化，并算一个任务集的**有效题量**。

In [ ]:
def binary_entropy(p):
    if p <= 0 or p >= 1:
        return 0.0
    return -(p * math.log2(p) + (1 - p) * math.log2(1 - p))

for p in [0.0, 0.05, 0.2, 0.3, 0.5, 0.7, 0.95, 1.0]:
    bar = '█' * int(binary_entropy(p) * 40)
    print(f'  成功率 {p:5.0%}  信息量 {binary_entropy(p):.3f} bit  {bar}')

assert binary_entropy(0.5) == 1.0
assert binary_entropy(0.0) == binary_entropy(1.0) == 0.0
assert binary_entropy(0.3) > binary_entropy(0.05)
print('\n✅ 30%-70% 区间内信息量 ≥ 0.88 bit，接近最大值；')
print('   而 5% 或 95% 的题只剩 0.29 bit——七成的测量预算被浪费掉了。')

In [ ]:
def effective_task_count(pass_rates):
    """有效题量 = 各题信息量之和 / 1 bit。衡量「名义 N 道题里，实际有多少道在干活」。"""
    return sum(binary_entropy(p) for p in pass_rates)

rng = np.random.default_rng(3)
# 三种任务集：饱和（大家都过）、地狱（大家都不过）、健康（难度有梯度）
saturated = np.clip(rng.normal(0.93, 0.05, 300), 0, 1)
hellish   = np.clip(rng.normal(0.04, 0.03, 300), 0, 1)
healthy   = np.clip(rng.uniform(0.15, 0.85, 300), 0, 1)

for name, rates in [('饱和 (均值0.93)', saturated), ('地狱 (均值0.04)', hellish),
                    ('健康 (均匀0.15-0.85)', healthy)]:
    eff = effective_task_count(rates)
    print(f'{name:<22} 名义 300 题 → 有效 {eff:6.1f} 题（利用率 {eff/300:5.1%}）')

assert effective_task_count(healthy) > 2 * effective_task_count(saturated)
print('\n✅ 同样 300 道题，健康任务集的测量效率是饱和任务集的两倍以上。')
print('   这就是「基准饱和了就该换基准」的定量版本——不是情怀问题，是信息论问题。')

## 4 · n-gram 污染检测器

最朴素也最实用的污染信号：任务文本与训练语料的 n-gram 重合率。
纯 Python 实现，没有依赖。

In [ ]:
def ngrams(text, n=13):
    toks = re.findall(r'\w+', text.lower())
    return {tuple(toks[i:i + n]) for i in range(max(0, len(toks) - n + 1))}

def contamination_score(task_text, corpus_texts, n=13):
    """返回任务文本里有多大比例的 n-gram 能在语料中找到。"""
    task_ngrams = ngrams(task_text, n)
    if not task_ngrams:
        return 0.0
    corpus_ngrams = set()
    for t in corpus_texts:
        corpus_ngrams |= ngrams(t, n)
    return len(task_ngrams & corpus_ngrams) / len(task_ngrams)

CORPUS = [
    'when the parser encounters a nested list inside a table cell it raises an '
    'index error because the cell width is computed before the nested content is expanded',
    'the recommended fix is to defer width computation until after all nested '
    'structures have been fully expanded and flattened into the layout tree',
]
clean_task = ('the exporter drops trailing whitespace in code blocks which breaks '
              'doctest output comparison for users who rely on exact formatting rules')
dirty_task = ('when the parser encounters a nested list inside a table cell it raises an '
              'index error because the cell width is computed before the nested content is expanded')

c_clean = contamination_score(clean_task, CORPUS, n=8)
c_dirty = contamination_score(dirty_task, CORPUS, n=8)
print(f'干净任务的 8-gram 重合率: {c_clean:.1%}')
print(f'污染任务的 8-gram 重合率: {c_dirty:.1%}')
assert c_clean < 0.05 and c_dirty > 0.9
print('\n✅ n-gram 检测能抓「逐字重合」，但抓不到「语义重合」（改写过的同一道题）。')
print('   更强的做法是嵌入相似度 + 「不给题面只给文件名，看模型能不能做对」的行为学检验。')

## 5 · 时间截断检验：训练截止前后的分数断崖

比 n-gram 更有说服力的污染证据：把任务按**创建时间**排序，
看模型的成功率在训练截止日期附近有没有断崖。

In [ ]:
def time_cutoff_test(dates, scores, cutoff):
    """返回 (截止前成功率, 截止后成功率, 差值)。差值显著为正 = 污染嫌疑。"""
    dates, scores = np.asarray(dates), np.asarray(scores, dtype=float)
    before, after = scores[dates < cutoff], scores[dates >= cutoff]
    return before.mean(), after.mean(), before.mean() - after.mean()

rng = np.random.default_rng(5)
n = 600
dates = rng.uniform(0, 24, size=n)              # 过去 24 个月
CUTOFF = 14.0                                    # 训练截止在第 14 个月

# 场景 A：被污染的模型——截止前的任务它"见过"
p_contaminated = np.where(dates < CUTOFF, 0.62, 0.34)
scores_contaminated = (rng.random(n) < p_contaminated).astype(float)
# 场景 B：干净的模型——成功率与任务创建时间无关
scores_clean = (rng.random(n) < 0.40).astype(float)

for name, sc in [('污染嫌疑模型', scores_contaminated), ('干净模型', scores_clean)]:
    b, a, d = time_cutoff_test(dates, sc, CUTOFF)
    # 两比例差的标准误
    nb, na = (dates < CUTOFF).sum(), (dates >= CUTOFF).sum()
    se = math.sqrt(b * (1 - b) / nb + a * (1 - a) / na)
    z = d / se
    print(f'{name:<14} 截止前 {b:.1%} | 截止后 {a:.1%} | 差 {d:+.1%} | z = {z:+.2f}')

b, a, d = time_cutoff_test(dates, scores_contaminated, CUTOFF)
assert d > 0.15, '污染场景下应有明显断崖'
b2, a2, d2 = time_cutoff_test(dates, scores_clean, CUTOFF)
assert abs(d2) < 0.10, '干净场景下不应有系统性断崖'
print('\n✅ 时间截断检验的强项：它不需要访问训练语料，只需要任务的创建时间。')
print('   弱点：任务难度本身可能随时间变化（新代码更复杂），需要用同期的对照任务集控制。')

## 6 · 任务集规模与分辨力：要区分 3 个点，需要多少道题

agentic 基准最痛的现实约束：任务少。用两比例检验反推所需样本量。

In [ ]:
def required_n(p1, p2, alpha=0.05, power=0.8):
    """两独立比例检验所需的每组样本量（正态近似）。"""
    z_a, z_b = 1.959963985, 0.8416212336        # 双侧 alpha=0.05 / power=0.8
    p_bar = (p1 + p2) / 2
    num = (z_a * math.sqrt(2 * p_bar * (1 - p_bar)) + z_b * math.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2
    return math.ceil(num / (p1 - p2) ** 2)

print(f"{'对比':<26}{'每组所需任务数':>16}")
for p1, p2 in [(0.40, 0.50), (0.40, 0.45), (0.40, 0.43), (0.40, 0.41)]:
    print(f'{p1:.0%} vs {p2:.0%}{"":<16}{required_n(p1, p2):>14,}')

n_needed = required_n(0.40, 0.43)
print(f'\nSWE-bench Verified 只有 500 道题，而区分 40% vs 43% 需要每组约 {n_needed:,} 道。')
assert n_needed > 500
print('→ 结论很硬：**在 500 道题的基准上，3 个点的差距在统计上不可分辨**。')
print('  榜单上挤在几个点内的名次，绝大部分是噪声。')
print('\n两条出路（04 模块展开）：')
print('  ① 配对设计——同一批任务上比较两个 agent，消掉任务难度这个最大的方差源；')
print('  ② 多次重复——每个任务跑 k 次，用任务内平均代替 0/1，降低单任务方差。')

In [ ]:
def required_n_paired(p_discordant, delta, alpha=0.05, power=0.8):
    """配对设计（McNemar）所需任务数：只有「一个对一个错」的不一致对携带信息。
    p_discordant: 不一致对的比例；delta: 两个 agent 的成功率之差。"""
    z_a, z_b = 1.959963985, 0.8416212336
    return math.ceil(((z_a + z_b) ** 2 * p_discordant) / (delta ** 2))

print('配对设计（同一批任务同时跑两个 agent）：')
for pd in [0.10, 0.20, 0.30]:
    print(f'  不一致对占比 {pd:.0%} → 区分 3 个点需要 {required_n_paired(pd, 0.03):>6,} 道任务')
gain = required_n(0.40, 0.43) / required_n_paired(0.20, 0.03)
print(f'\n配对设计相对独立设计的样本量节省：约 {gain:.1f} 倍')
assert gain > 1.0
print('✅ 这就是「同一批任务、同一套 harness、同时跑两个 agent」为什么是 agent 评测的默认设计。')

## ✏️ 练习 1：任务集健康度评分

实现 `task_set_health(pass_rates)`：返回一个 0–1 的健康度 = **有效题量 / 名义题量**
（即各题二元熵的平均）。用它比较三种任务集。

In [ ]:
def task_set_health(pass_rates):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert abs(task_set_health([0.5] * 10) - 1.0) < 1e-12
assert abs(task_set_health([0.0, 1.0, 0.0, 1.0])) < 1e-12
h_healthy, h_sat = task_set_health(healthy), task_set_health(saturated)
assert h_healthy > 0.8 and h_sat < 0.5
print(f'健康任务集 {h_healthy:.1%} | 饱和任务集 {h_sat:.1%}')
print('✅ 练习 1 通过：健康度掉到 50% 以下，说明该换基准或加难题了。')

## ✏️ 练习 2：不可能任务的正确处理率

实现 `impossible_task_score(responses)`：输入一批对**不可能任务**的响应
（每条形如 `{'abstained': bool, 'fabricated': bool}`），返回
`(正确放弃率, 编造率)`。正确放弃 = `abstained and not fabricated`。

In [ ]:
def impossible_task_score(responses):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
resp = [{'abstained': True,  'fabricated': False},
        {'abstained': False, 'fabricated': True},
        {'abstained': False, 'fabricated': True},
        {'abstained': True,  'fabricated': False}]
ab, fab = impossible_task_score(resp)
assert abs(ab - 0.5) < 1e-12 and abs(fab - 0.5) < 1e-12
all_good = [{'abstained': True, 'fabricated': False}] * 5
assert impossible_task_score(all_good) == (1.0, 0.0)
print(f'正确放弃率 {ab:.0%} | 编造率 {fab:.0%}')
print('✅ 练习 2 通过：没有不可能任务的基准，会系统性奖励编造——')
print('   而编造恰恰是 agent 在真实产品里最危险的失败模式。')

## ✏️ 练习 3：污染嫌疑的组合判据

实现 `contamination_flag(ngram_overlap, time_gap_delta, name_recall)`：
三个信号任意<strong>两个</strong>超阈值就标记为嫌疑。阈值：
`ngram_overlap > 0.3`、`time_gap_delta > 0.12`、`name_recall > 0.5`
（`name_recall` = 不给题面、只给文件名时模型说出私有符号名的比例）。
返回 `(是否嫌疑, 触发的信号列表)`。

In [ ]:
def contamination_flag(ngram_overlap, time_gap_delta, name_recall):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
flag, sig = contamination_flag(0.45, 0.20, 0.10)
assert flag is True and set(sig) == {'ngram_overlap', 'time_gap_delta'}
flag2, sig2 = contamination_flag(0.45, 0.02, 0.10)
assert flag2 is False and sig2 == ['ngram_overlap']
flag3, sig3 = contamination_flag(0.9, 0.9, 0.9)
assert flag3 is True and len(sig3) == 3
print('单一信号不足以定罪：', contamination_flag(0.45, 0.02, 0.10))
print('两个信号同时触发：  ', contamination_flag(0.45, 0.20, 0.10))
print('✅ 练习 3 通过：单个污染信号都有各自的假阳性来源——')
print('   n-gram 会被通用样板文本触发，时间断崖会被难度漂移触发，需要交叉印证。')

## ✏️ 练习 4：给定预算下的任务集配比

实现 `allocate_tasks(budget_minutes, cost_per_task, min_per_bucket)`：
`cost_per_task` 是 `{难度: 每题分钟数}`，按「每一分钟买到的信息量最大」贪心分配，
每档至少 `min_per_bucket` 道。假设各档难度对应的成功率为
`{'easy': 0.85, 'medium': 0.5, 'hard': 0.2}`，信息量用二元熵。
返回 `{难度: 题数}`。

In [ ]:
PASS_BY_BUCKET = {'easy': 0.85, 'medium': 0.5, 'hard': 0.2}

def allocate_tasks(budget_minutes, cost_per_task, min_per_bucket):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
cost = {'easy': 2.0, 'medium': 3.0, 'hard': 20.0}
alloc = allocate_tasks(600, cost, min_per_bucket=5)
assert set(alloc) == {'easy', 'medium', 'hard'}
assert all(v >= 5 for v in alloc.values())
spent = sum(alloc[k] * cost[k] for k in alloc)
assert spent <= 600 + 1e-9, f'超预算: {spent}'
# medium 的「每分钟信息量」最高（1.0 bit / 6 min），应拿到最多的追加名额
assert alloc['medium'] > alloc['hard']
info = sum(alloc[k] * binary_entropy(PASS_BY_BUCKET[k]) for k in alloc)
print('分配结果：', alloc, f'| 耗时 {spent:.0f} 分钟 | 总信息量 {info:.1f} bit')
print('✅ 练习 4 通过：medium 每分钟买到 0.33 bit（最高），hard 只有 0.036 bit（最低）——')
print('   但保底名额仍然必须给，否则任务集会失去对强模型的区分度（这是「保底 vs 效率」的经典取舍）。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def task_set_health(pass_rates):
    rates = list(pass_rates)
    if not rates:
        return 0.0
    return sum(binary_entropy(p) for p in rates) / len(rates)

In [ ]:
# 练习 2 参考答案
def impossible_task_score(responses):
    n = len(responses)
    if n == 0:
        return (0.0, 0.0)
    ab = sum(1 for r in responses if r['abstained'] and not r['fabricated']) / n
    fab = sum(1 for r in responses if r['fabricated']) / n
    return (ab, fab)

In [ ]:
# 练习 3 参考答案
def contamination_flag(ngram_overlap, time_gap_delta, name_recall):
    signals = []
    if ngram_overlap > 0.3:
        signals.append('ngram_overlap')
    if time_gap_delta > 0.12:
        signals.append('time_gap_delta')
    if name_recall > 0.5:
        signals.append('name_recall')
    return (len(signals) >= 2, signals)

In [ ]:
# 练习 4 参考答案
def allocate_tasks(budget_minutes, cost_per_task, min_per_bucket):
    alloc = {k: min_per_bucket for k in cost_per_task}
    remaining = budget_minutes - sum(min_per_bucket * c for c in cost_per_task.values())
    if remaining < 0:
        raise ValueError('预算不足以覆盖每档的保底名额')
    # 每分钟买到的信息量，从高到低贪心
    order = sorted(cost_per_task, key=lambda k: -binary_entropy(PASS_BY_BUCKET[k]) / cost_per_task[k])
    for k in order:
        n_extra = int(remaining // cost_per_task[k])
        alloc[k] += n_extra
        remaining -= n_extra * cost_per_task[k]
    return alloc

---
## 🧪 真实工程胶囊：接入三个真实基准的最短路径

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════
# A. SWE-bench Verified：查看任务、检查污染窗口
# ══════════════════════════════════════════════════════════════════
from datasets import load_dataset
ds = load_dataset("princeton-nlp/SWE-bench_Verified", split="test")
print(len(ds))                       # 500
ex = ds[0]
# 污染自检：任务对应 PR 的创建时间 vs 你的模型训练截止时间
print(ex["created_at"], ex["repo"], ex["version"])
# 清洗规则（自建任务集时必做）：剥离 issue 评论、后续 commit message、PR 描述
import re
clean = re.split(r"\n\s*(?:Comment|Reply|Fix|Patch)\b", ex["problem_statement"])[0]

# ══════════════════════════════════════════════════════════════════
# B. tau-bench：注意用户模拟器是评测的一部分
# ══════════════════════════════════════════════════════════════════
# pip install tau-bench   (github.com/sierra-research/tau-bench)
# python run.py --agent-strategy tool-calling \
#     --env retail --model claude-sonnet-5 --model-provider anthropic \
#     --user-model claude-sonnet-5 --user-model-provider anthropic \
#     --num-trials 8            # ← 8 次重复才能算 pass^k
# 报告时必须写清 user-model 是谁：换了用户模拟器 = 换了基准，分数不可比。

# ══════════════════════════════════════════════════════════════════
# C. WebArena：自托管快照才是可复现的关键
# ══════════════════════════════════════════════════════════════════
# 官方提供各站点的 docker 镜像（shopping / reddit / gitlab / cms / map）。
# 复现清单（缺一项就不可复现）：
#   1. 镜像 digest（不是 tag——tag 会被覆盖）
#   2. 每次任务开始前 reset 到快照（否则前一个任务的写操作会污染后一个）
#   3. 固定浏览器版本与视窗尺寸（页面布局变化会让基于坐标的动作失效）
#   4. 记录 validator 的版本——validator 是手写代码，它自己也会被修 bug

# ══════════════════════════════════════════════════════════════════
# D. 自建任务集的最小 schema（照抄即可）
# ══════════════════════════════════════════════════════════════════
TASK_SCHEMA = {
  "task_id": "str, 稳定不变",
  "prompt": "str, 已剥离解法泄漏",
  "env_snapshot": "str, 镜像 digest 或数据快照哈希",
  "scorer": "tests | final_state | validator | exact_match",
  "success_check": "必须达成的条件",
  "regression_check": "必须保持不变的条件（对应 PASS_TO_PASS）",
  "impossible": "bool, 5%-15% 的任务应为 True",
  "meta": {"n_tools_expected": 3, "human_minutes": 12, "subsystem": "billing",
           "created_at": "2026-03-01"},
}
'''
print(RECIPE)

### 小结

| 你学到的 | 一句话 | 用在哪 |
|---|---|---|
| 按环境分类 | 环境决定判分方式，判分方式决定分数能支持什么结论 | 选基准 / 自建任务集 |
| SWE-bench 的三个数字 | 2294 全量 / 500 人工审核 / 300 轻量；Verified 的存在证明「坏题是常态」 | 报告主分数用 Verified |
| 终态匹配与 pass^k | 客服型 agent 的价值在可靠性，pass@k 报出来没有意义 | 04 模块 |
| 程序化 validator | 每题一个手写函数——validator 自己也需要被测试 | 02 模块 |
| 四种共同病灶 | 污染 / 坏题 / 环境漂移 / 饱和与 harness 过拟合 | 读任何评测报告 |
| 信息论视角 | 成功率推向 30–70% 才有区分度；500 题分辨不了 3 个点 | 04 模块 |

下一模块：**02 · 结果判分与部分得分**——把「怎么判对错」从一句话展开成一套工程，
包括判分器自身的假阳/假阴、部分得分会不会改变模型排序、以及 rubric checkpoint 怎么设计。